# nivult-v2 — GIRO C

**Prima di lanciare: Runtime → Disconnetti ed elimina runtime, poi ricollega una A100.**
Le 24 ore contano per MACCHINA: una macchina nuova riparte da 24, una vecchia ti lascia quello che le resta.

Dataset nuovo: **812.094 righe, firma `812094:a3c7d0a3b2c6`**. La cella 4 lo verifica prima di spendere un minuto di GPU.

**I file si caricano UNA volta sola.** La cella 2 li mette su Drive dopo il caricamento, e alla macchina successiva se li riprende da là: non devi ricaricare 315 MB a ogni riavvio. I quattro file piccoli (gli script e il golden) si ricaricano ogni volta apposta — pesano nulla e così non si rischia di addestrare con una versione vecchia.

L'addestramento dura ~26 ore, quindi si spezzerà su più macchine. Il checkpoint sta su Drive: si riprende da solo lanciando di nuovo la cella 6.

Due cose che l'esame farà emergere e che NON sono novità: `lead` è allo 0,44% del train contro l'11,4% delle 280 a mano (servirebbe una fonte di `lead`, non un altro giro), e il remoto è per il 93% esempi di stima.

In [ ]:
# 1) macchina e cartella di uscita
MODELLO = "Qwen/Qwen3.5-9B"
EPOCHE = 1
MAX_LEN = 1024
import torch
VRAM = torch.cuda.get_device_properties(0).total_memory/1e9
BS = 8 if VRAM >= 70 else 4
ACC = 2 if VRAM >= 70 else 4
print('GPU', torch.cuda.get_device_name(0), round(VRAM), 'GB -> lotto', BS, 'x', ACC)
# «-c» apposta: il LoRA e l'esame del giro B restano intatti in qwen3.5-9b-b,
# che e' l'unico modello di cui conosciamo davvero i numeri
NOME = MODELLO.split('/')[-1].lower() + '-c'
from google.colab import drive
drive.mount('/content/drive')
import os; OUT = f'/content/drive/MyDrive/nivult-v2/{NOME}'; os.makedirs(OUT, exist_ok=True); print('cartella', OUT)

In [ ]:
# 2) file — i due grossi si caricano UNA volta sola, poi vivono su Drive
from google.colab import files
import os, shutil

DRIVE_FILE = '/content/drive/MyDrive/nivult-v2/file'
os.makedirs(DRIVE_FILE, exist_ok=True)

# I grossi, con la dimensione ESATTA attesa: su Drive c'e' ancora quello del
# giro B (225 MB) e ripescarlo per sbaglio vorrebbe dire addestrare sul
# dataset vecchio senza accorgersene.
GROSSI = {'sft-train.jsonl.gz': 315110103, 'dataset-esame-v2.jsonl.gz': 87103528}
# I piccoli si ricaricano SEMPRE: pesano nulla e cosi' non si addestra mai
# con uno script vecchio. addestra_v2.py e esame_v2.py sono cambiati l'11/09.
PICCOLI = ('prompt_v2.py', 'addestra_v2.py', 'esame_v2.py', 'dataset-golden-v1.jsonl.gz')

for f, atteso in GROSSI.items():
    if os.path.exists(f) and os.path.getsize(f) != atteso:
        os.remove(f)
    d = f'{DRIVE_FILE}/{f}'
    if not os.path.exists(f) and os.path.exists(d) and os.path.getsize(d) == atteso:
        print('lo riprendo da Drive, non ricaricarlo:', f); shutil.copy(d, f)

manca = [f for f in list(GROSSI) + list(PICCOLI) if not os.path.exists(f)]
if manca:
    print('CARICA QUESTI:', manca)
    files.upload()

# da ora stanno su Drive: la prossima macchina non te li richiede
for f, atteso in GROSSI.items():
    d = f'{DRIVE_FILE}/{f}'
    if os.path.exists(f) and os.path.getsize(f) == atteso and (not os.path.exists(d) or os.path.getsize(d) != atteso):
        print('lo salvo su Drive per la prossima volta:', f); shutil.copy(f, d)

print({f: os.path.getsize(f)//1024//1024 for f in list(GROSSI) + list(PICCOLI) if os.path.exists(f)}, 'MB')
for f, atteso in GROSSI.items():
    assert os.path.exists(f) and os.path.getsize(f) == atteso, f'{f}: manca o e\' la versione sbagliata (attesi {atteso} byte)'
for f in PICCOLI:
    assert os.path.exists(f), f'manca {f}'
print('tutti i file ci sono, e sono quelli giusti')

In [ ]:
# 3) dipendenze
!pip -q install -U unsloth trl datasets peft accelerate
import torch; print('GPU', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9), 'GB')

In [ ]:
# 4) la firma del dataset, PRIMA di spendere GPU
# Il giro B e' stato addestrato bene e giudicato male per un difetto che
# nessuno vedeva. Qui si controlla l'unica cosa che identifica il dataset.
import gzip, hashlib
TRAIN = 'sft-train.jsonl.gz'
ATTESA = '812094:a3c7d0a3b2c6'

def firma(p):
    h = hashlib.sha1(); n = 0
    with gzip.open(p, 'rb') as f:
        while True:
            b = f.read(1 << 20)
            if not b:
                break
            if n == 0:
                h.update(b)
            n += b.count(b'\n')
    return f'{n}:{h.hexdigest()[:12]}'

f = firma(TRAIN)
print('firma trovata:', f)
assert f == ATTESA, f'DATASET SBAGLIATO: atteso {ATTESA}. Questo e\' il file del giro B?'
print('OK, e\' il dataset del giro C: 812.094 righe')

In [ ]:
# 5) PROVA su 300 righe (3-5 minuti). Se questa passa, passa anche il giro lungo.
# Deve stampare «LoRA: 200 tensori lora_B, di cui 72 sull'attenzione lineare»:
# se dice 128 e 0, i mixer lineari non hanno ricevuto l'adattatore e il giro
# lungo sarebbe uguale a quello di prima.
!rm -rf /content/prova-v2 && python addestra_v2.py --modello "$MODELLO" --train "$TRAIN" --out /content/prova-v2 --epoche 1 --bs $BS --accumulo 1 --max-len $MAX_LEN --max-righe 300 2>&1 | grep -viE "^warning|warn\(" | tail -14
import os, json
assert os.path.exists('/content/prova-v2/rapporto-addestramento.json'), 'la prova NON e\' arrivata in fondo: mandami le righe qui sopra'
print('PROVA OK:', json.load(open('/content/prova-v2/rapporto-addestramento.json')))

In [ ]:
# 6) ADDESTRAMENTO (~26 ore, riprende da solo dal checkpoint su Drive)
# Se la macchina muore: macchina nuova, celle 1-2-3, poi di nuovo QUESTA.
# Riparte dall'ultimo checkpoint, non da zero.
!python addestra_v2.py --modello "$MODELLO" --train "$TRAIN" --out "$OUT" --epoche $EPOCHE --bs $BS --accumulo $ACC --max-len $MAX_LEN

In [ ]:
# 7) L'ESAME: qui si decide
# Deve stampare «LoRA: N tensori lora_B, somma |B| = ...» con la somma DIVERSA
# da zero. Se e' zero si ferma da solo: vorrebbe dire misurare Qwen nudo,
# ed e' esattamente l'errore che ha fatto bocciare il primo tentativo.
!python esame_v2.py --modello "$MODELLO" --adapter "$OUT/lora" --golden dataset-golden-v1.jsonl.gz --esame dataset-esame-v2.jsonl.gz --out "$OUT/esame-v2.json" --bs 16
import json
r = json.load(open(f'{OUT}/esame-v2.json'))
print('\nCANCELLO:', json.dumps(r['cancello'], indent=1))
print('banco dei codici:', r['codici'].get('banco'))
print('dichiarati:', {k: v.get('accuratezza') for k, v in r['dichiarati'].items()})
print('\nconfronto col giro B: famiglia a mano 0,896 | codici 0,725 | seniority 0,645 | contratto 0,656 | remoto 0,980')

In [ ]:
# 8) pacchetto da scaricare: adattatori LoRA + rapporti (pochi MB)
!cd "$OUT" && zip -q -r /content/nivult-v2-$NOME.zip lora esame-v2.json rapporto-addestramento.json
from google.colab import files; files.download(f'/content/nivult-v2-{NOME}.zip')